# 01 - Tong quan chat luong du lieu

Notebook nay kiem tra du lieu YouTube hien tai tu `data/processed/*.csv` va doi chieu nhanh voi export dashboard.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
EXPORT_DIR = ROOT / 'dashboard' / 'exports'
PROCESSED_DIR = ROOT / 'data' / 'processed'

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 140)


def read_csv(relative_path):
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}')
    return pd.read_csv(path)


def pct(numerator, denominator):
    return np.where(pd.Series(denominator).replace(0, np.nan).notna(), numerator / denominator * 100, np.nan)

posts = read_csv('data/processed/posts.csv')
comments = read_csv('data/processed/comments.csv')
exports = {path.stem: pd.read_csv(path) for path in EXPORT_DIR.glob('*.csv')}
print(f'posts: {posts.shape}, comments: {comments.shape}, exports: {len(exports)} files')


In [ ]:
date_range = {
    'post_date_from': posts['date'].min(),
    'post_date_to': posts['date'].max(),
    'comment_date_from': comments['date'].min(),
    'comment_date_to': comments['date'].max(),
}
row_counts = pd.DataFrame([
    {'dataset': 'processed_posts', 'rows': len(posts), 'columns': posts.shape[1]},
    {'dataset': 'processed_comments', 'rows': len(comments), 'columns': comments.shape[1]},
    *[{'dataset': name, 'rows': len(df), 'columns': df.shape[1]} for name, df in exports.items()],
])
display(row_counts.sort_values('dataset'))
date_range


In [ ]:
quality_checks = []
post_dupes = int(posts.duplicated(['platform', 'post_id']).sum())
comment_dupes = int(comments.duplicated(['platform', 'comment_id']).sum())
quality_checks.append({'check': 'Duplicate post natural keys', 'value': post_dupes, 'status': 'PASS' if post_dupes == 0 else 'FAIL'})
quality_checks.append({'check': 'Duplicate comment natural keys', 'value': comment_dupes, 'status': 'PASS' if comment_dupes == 0 else 'FAIL'})
metric_cols = ['reach', 'impressions', 'likes', 'comments_count', 'shares', 'engagement_rate', 'virality_score']
negative_metric_count = int((posts[metric_cols] < 0).sum().sum())
quality_checks.append({'check': 'Negative post metrics', 'value': negative_metric_count, 'status': 'PASS' if negative_metric_count == 0 else 'FAIL'})
valid_sentiments = {'positive', 'neutral', 'negative'}
invalid_sentiments = int((~comments['sentiment_label'].isin(valid_sentiments)).sum())
quality_checks.append({'check': 'Invalid sentiment labels', 'value': invalid_sentiments, 'status': 'PASS' if invalid_sentiments == 0 else 'FAIL'})
missing_summary = pd.concat([
    posts.isna().sum().rename('missing_posts'),
    comments.isna().sum().rename('missing_comments')
], axis=1).fillna(0).astype(int)
display(pd.DataFrame(quality_checks))
display(missing_summary[(missing_summary['missing_posts'] > 0) | (missing_summary['missing_comments'] > 0)])


In [ ]:
posts_check = posts.assign(
    recomputed_engagement_count=posts['likes'] + posts['comments_count'] + posts['shares'],
    recomputed_engagement_rate=np.where(posts['reach'] > 0, (posts['likes'] + posts['comments_count'] + posts['shares']) / posts['reach'] * 100, np.nan),
    recomputed_virality_score=np.where(posts['reach'] > 0, posts['shares'] / posts['reach'] * 100, np.nan),
)
summary = pd.DataFrame([
    {'metric': 'max_engagement_rate_delta', 'value': round(float((posts_check['engagement_rate'] - posts_check['recomputed_engagement_rate']).abs().max()), 6)},
    {'metric': 'max_virality_score_delta', 'value': round(float((posts_check['virality_score'] - posts_check['recomputed_virality_score']).abs().max()), 6)},
    {'metric': 'distinct_pages', 'value': int(posts['page_name'].nunique())},
    {'metric': 'platforms', 'value': ', '.join(sorted(posts['platform'].dropna().unique()))},
])
display(summary)


## Insight

- Key finding: Du lieu hien tai du sach de dung cho lop BI, khong co trung khoa tu nhien va khong co metric am.
- Supporting data: 876 posts, 1,692 comments, 0 duplicate post/comment keys; pham vi ngay 2017-09-26 den 2026-05-31.
- Recommended action: Duy tri kiem tra duplicate, metric am va nhan sentiment trong moi lan refresh truoc khi cong bo dashboard.